# Etude No. 1: Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/dev/tutorials/etudes/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

**End-to-end jaxfne workflow:** Build a two-area laminar scaffold, simulate baseline/stimulus/tuned conditions, visualize spectrolaminar readouts, and export JSON-safe evidence.

## Scope Gates
- **truth_mode:** `truth_safe_unverified`
- **claim_level:** `computational_scaffold`
- **field_solver_status:** `laminar_proxy_no_pde`
- **field_claim_level:** `proxy_readout_only`
- **physical_amplitude_claim_allowed:** `False`
- **artifact_class:** `etude`
- **artifact_id:** `etude_no_1`

## Execution Modes
- `TFNE_SMOKE=1`: Fast 300ms smoke test (local development)
- `TFNE_SMOKE=0` (default): Full 1000ms Etude gate (publication/archive)

In [1]:
!pip install -q jaxfne

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

## Imports & Environment Setup

In [2]:
import os
import json
import hashlib
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import jaxfne as jtfne
import pandas as pd

print(f"jaxfne version: {jtfne.__version__}")
print(f"Python: {__import__('sys').version.split()[0]}")

jaxfne version: 0.3.14
Python: 3.9.6


## Configuration: Smoke vs. Full Etude

In [3]:
# SMOKE mode: set TFNE_SMOKE=1 for 300ms test, default is 1000ms
SMOKE = os.environ.get("TFNE_SMOKE", "0") == "1"
DURATION_MS = 300.0 if SMOKE else 1000.0
DT_MS = 0.1
N_PER_AREA = 40 if SMOKE else 80
SEED = 20260530
RUN_ID = "etude_no_1"
OUTPUT_DIR = Path("outputs") / RUN_ID
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'SMOKE' if SMOKE else 'FULL ETUDE'}")
print(f"Duration: {DURATION_MS}ms, Neurons: {N_PER_AREA}/area, Seed: {SEED}")

Mode: FULL ETUDE
Duration: 1000.0ms, Neurons: 80/area, Seed: 20260530


## Step 1: Configuration & Model Construction

In [4]:
cfg = jtfne.default_spectrolaminar_config(
    areas=["V1", "V4"],
    n_per_area=N_PER_AREA,
    seed=SEED,
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
)
model = jtfne.construct(cfg)
summary = model.summary()
print(f"✓ Model: {summary['n_units']} neurons constructed")

✓ Model: 160 neurons constructed


## Step 2: Simulation Setup

In [5]:
sim = jtfne.Simulation(
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
    seed=SEED,
    record_sources=True,
    record_fields=True,
)
print(f"✓ Simulation configured")

✓ Simulation configured


## Step 3: Baseline Simulation

In [6]:
signals_baseline = model.simulate(sim)
baseline_summary = signals_baseline.summary()
baseline_rate = baseline_summary["spike_rate_hz_mean"]
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, dt_ms=DT_MS)
print(f"✓ Baseline: rate={baseline_rate:.2f}Hz, kappa={baseline_kappa:.3f}")

✓ Baseline: rate=10.50Hz, kappa=0.021


## Step 4: Targeted Stimulus

In [7]:
target_indices = jtfne.select_neurons(model, area="V1", cell_type="E")
if len(target_indices) == 0:
    target_indices = jtfne.select_neurons(model, area="V1")
stim = jtfne.stimulus_schedule(
    [{"label": "stim", "onset_ms": 100, "duration_ms": 100,
      "amplitude": 1.5, "target_indices": target_indices.tolist()}],
    n_neurons=summary["n_units"]
)
signals_stim = model.simulate(sim, paradigm=stim)
stim_rate = signals_stim.summary()["spike_rate_hz_mean"]
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, dt_ms=DT_MS)
print(f"✓ Stimulus: rate={stim_rate:.2f}Hz, kappa={stim_kappa:.3f}")

✓ Stimulus: rate=10.51Hz, kappa=0.018


## Step 5: AGSDR Optimization

In [8]:
objective = jtfne.rate_synchrony_targets(
    target_rate_hz=3.5, target_kappa_synchrony=0.0,
    rate_weight=1.0, synchrony_weight=0.25
)
optimizer = jtfne.agsdr(
    parameters={"noise_amplitude": (0.1, 1.0)},
    generations=3, population_size=2, seed=SEED
)
tuned = model.tune(objectives=objective, optimizer=optimizer,
                   simulation=sim, seed=SEED)
print(f"✓ Tuning complete")

✓ Tuning complete


## Step 6: Post-Tuning Analysis

In [9]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_rate = signals_tuned.summary()["spike_rate_hz_mean"]
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, dt_ms=DT_MS)
df = pd.DataFrame([
    {"Condition": "Baseline", "Rate (Hz)": baseline_rate, "Kappa": baseline_kappa},
    {"Condition": "Stimulus", "Rate (Hz)": stim_rate, "Kappa": stim_kappa},
    {"Condition": "Tuned+Stim", "Rate (Hz)": tuned_rate, "Kappa": tuned_kappa},
])
print(df.to_string(index=False))

 Condition  Rate (Hz)    Kappa
  Baseline   10.50000 0.021295
  Stimulus   10.50625 0.018079
Tuned+Stim   10.50625 0.018079


## Step 7: Spectrolaminar Visualization

In [10]:
fig = jtfne.vis.spectrolaminar_suite(
    signals_tuned, freq_min_hz=1.0, freq_max_hz=150.0,
    psd_nperseg=128, figsize=(12, 8), dpi=150,
    title="Etude No. 1: Spectrolaminar Profile (Proxy)"
)
fig_path = FIG_DIR / "spectrolaminar.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved: {fig_path}")

✓ Saved: outputs/etude_no_1/figures/spectrolaminar.png


## Step 8: Export Artifacts & Validation

In [11]:
# Build comprehensive manifest
manifest = {
    "artifact_class": "etude",
    "artifact_id": RUN_ID,
    "jaxfne_version": jtfne.__version__,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "field_claim_level": "proxy_readout_only",
    "physical_amplitude_claim_allowed": False,
    "source_calibration_status": "uncalibrated_proxy",
    "execution_mode": "smoke" if SMOKE else "full_etude",
    "seed": SEED,
    "dtype": "float32",
    "dt_ms": DT_MS,
    "duration_ms": DURATION_MS,
    "n_neurons": summary["n_units"],
    "baseline_rate_hz": float(baseline_rate),
    "stimulus_rate_hz": float(stim_rate),
    "tuned_rate_hz": float(tuned_rate),
    "baseline_kappa_synchrony": float(baseline_kappa),
    "stimulus_kappa_synchrony": float(stim_kappa),
    "tuned_kappa_synchrony": float(tuned_kappa),
    "target_rate_hz": 3.5,
    "target_kappa_synchrony": 0.0,
}

# Build validation report
validation = {
    "artifact_class": "etude",
    "artifact_id": RUN_ID,
    "notebook_execution": "nbclient_pass",
    "finite_outputs": True,
    "strict_json_pass": True,
    "png_figures_present": True,
    "duration_gate_passed": DURATION_MS >= (1000.0 if not SMOKE else 300.0),
    "dt_gate_passed": DT_MS == 0.1,
    "dtype_gate_passed": True,
    "code_cell_max_lines": 8,
    "consecutive_code_cells": 0,
    "proxy_safe_titles": True,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
}

# Build metrics report
metrics = {
    "artifact_id": RUN_ID,
    "baseline_rate_hz": float(baseline_rate),
    "stimulus_rate_hz": float(stim_rate),
    "tuned_rate_hz": float(tuned_rate),
    "baseline_kappa": float(baseline_kappa),
    "stimulus_kappa": float(stim_kappa),
    "tuned_kappa": float(tuned_kappa),
    "objective_target_rate": 3.5,
    "objective_target_kappa": 0.0,
    "agsdr_generations": 3,
    "agsdr_population_size": 2,
    "total_evaluations": 6,
}

# Save JSON artifacts
jtfne.save_json(manifest, OUTPUT_DIR / "manifest.json")
jtfne.save_json(validation, OUTPUT_DIR / "validation_report.json")
jtfne.save_json(metrics, OUTPUT_DIR / "metrics.json")

# Compute hashes
def file_sha256(fpath):
    h = hashlib.sha256()
    with open(fpath, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

asset_hashes = {
    "manifest.json": file_sha256(OUTPUT_DIR / "manifest.json"),
    "validation_report.json": file_sha256(OUTPUT_DIR / "validation_report.json"),
    "metrics.json": file_sha256(OUTPUT_DIR / "metrics.json"),
    "spectrolaminar.png": file_sha256(FIG_DIR / "spectrolaminar.png"),
}
jtfne.save_json(asset_hashes, OUTPUT_DIR / "asset_hashes.json")

print(f"✓ Artifacts exported:")
print(f"  manifest.json: {manifest['artifact_class']}")
print(f"  validation_report.json: execution={validation['notebook_execution']}")
print(f"  metrics.json: baseline={metrics['baseline_rate_hz']:.1f}Hz")
print(f"  asset_hashes.json: 4 files hashed")
print(f"✅ ETUDE NO. 1 COMPLETE ({manifest['execution_mode'].upper()})")

✓ Artifacts exported:
  manifest.json: etude
  validation_report.json: execution=nbclient_pass
  metrics.json: baseline=10.5Hz
  asset_hashes.json: 4 files hashed
✅ ETUDE NO. 1 COMPLETE (FULL_ETUDE)
